In [1]:
import re
import nltk
import time
import string
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer



In [2]:
# Functions


def save_model_bundle(filepath, model_score, gatekeeper, models, gatekeeper_thresh, 
                      thresholds, nounVectorizer, gramVectorizer, charVectorizer, scaler):
    """Saves the trained models, vectorizers, scaler, and thresholds to a single file."""
    bundle = {
        'model_score': model_score,
        'gatekeeper': gatekeeper,
        'models': models,
        'gatekeeper_thresh': gatekeeper_thresh,
        'thresholds': thresholds,
        'nounVectorizer': nounVectorizer,
        'gramVectorizer': gramVectorizer,
        'charVectorizer': charVectorizer,
        'scaler': scaler
    }
    joblib.dump(bundle, filepath)
    print(f"Model bundle successfully saved to {filepath}")


def load_model_bundle(filepath):
    """Loads the model bundle and returns the components in their original order."""
    bundle = joblib.load(filepath)
    print(f"Model bundle loaded from {filepath}")
    return (
        bundle['model_score'],
        bundle['gatekeeper'],
        bundle['models'],
        bundle['gatekeeper_thresh'],
        bundle['thresholds'],
        bundle['nounVectorizer'],
        bundle['gramVectorizer'],
        bundle['charVectorizer'],
        bundle['scaler']
    )
    
def preprocess_text(series):
     # @somone to @
    lst = []
    lstn = []
    lstw  = []
    length = len(series)
    for i, text in enumerate(series):
        # mark @, # and links
        text = ' '.join(['_at_someone_' if '@' in word else word for word in text.split()])
        text = ' '.join(['_link_' if 'http' in word or 'www.' in word else word for word in text.split()])
        text = ' '.join(['_hashtag_' if '#' in word else word for word in text.split()])
        # remove all other punctuation
        text = ''.join([word.lower() for word in text if word not in string.punctuation.replace('@','').replace('#','').replace('_','')])

        # the real work, pos tagging to reduce Nouns and other things that don't matter
        tokens = []
        nouns = []
        weird = []
        for a in nltk.pos_tag(text.split()):


            if bool(re.search(r'[^\w\s]', a[0])):
                weird.append(nltk.PorterStemmer().stem(a[0]))

            # keep these as-is for reasons
            if '_link_' in a[0] or '_at_someone_' in a[0] or '_hashtag_' in a[0]:
                tokens.append(a[0])
                nouns.append(a[0])
            # lemmatize these to reduce parts of speach issues
            elif 'NN' in a[1] or 'JJ' in a[1] or 'RB' in a[1] or 'V' in a[1]:
                tokens.append(a[1])
                try:
                    nouns.append(a[0])
                except:
                    print('Recursion error',a)
            else:
                tokens.append(a[1])
                #nouns.append(nltk.PorterStemmer().stem(a[0]))
        lst.append(' '.join(tokens))
        lstn.append(' '.join(nouns))
        lstw.append(' '.join(weird))
        if i%100 == 0:
            print("\r{:.2f}% Complete".format(100*((i+1)/length), a), end="", flush=True)
    print("\r100.00% Complete".format(100*((i+1)/length)), end="", flush=True)
    return lst, lstn, lstw

def vectorize_data(df, nounVectorizer, gramVectorizer, charVectorizer, scaler):
    gV = gramVectorizer.transform(df['pos'])
    nV = nounVectorizer.transform(df['stems'])
    nC = charVectorizer.transform(df['comment_text'])
    transformed_data = hstack([gV, nV, nC])

    if scaler != None:
        scaler.fit(transformed_data)
    return transformed_data

def fit_vectorizers(df):
    # gram is based on the work I did in howRussian
    gramVectorizer =  TfidfVectorizer(ngram_range=(2,5)
                                        , min_df = 10
                                        , max_df = .7
                                        , stop_words = None
                                        , sublinear_tf=True
                                        , lowercase=False              # Protect POS tag case sensitivity
                                        , token_pattern=r'\S+'          # Tokenize strictly by space-separated tags/tokens
                                       )
    gramVectorizer.fit(df['pos'])
    
    
    
    # a simple vectorizer on the nouns to include specific words because reasons, ideal at word and bigram
    nounVectorizer = TfidfVectorizer(ngram_range=(1, 2)                
                                        , tokenizer=tokenize              
                                        , strip_accents='unicode'            
                                        , stop_words='english'               
                                        , sublinear_tf=True  # Replaces TF count with 1 + log(TF)
                                        , min_df=3                           
                                        , max_df=0.9                           
                                    )
    nounVectorizer.fit(df['stems'])


    # char_vec to catch weird mispellings...
    charVectorizer = TfidfVectorizer(ngram_range=(2, 6)               
                                        , analyzer='char'                   
                                        , sublinear_tf=True                 
                                        , strip_accents='unicode'           
                                        , min_df=3                          
                                        , max_features=50000                  
                                    )
    charVectorizer.fit(df[df['flag']==1]['punc'])

    
    gV = gramVectorizer.transform(df['pos'])
    nV = nounVectorizer.transform(df['stems'])
    nC = charVectorizer.transform(df['punc'])
    transformed_data = hstack([gV, nV, nC])

    # scaler = StandardScaler(with_mean=False)
    scaler = None
    # scaler.fit(transformed_data)
    return gramVectorizer, nounVectorizer, charVectorizer, scaler

def set_threshold(model = None, models = None, col = None, df=None, Xt=None):
    testing_gate = pd.DataFrame()
    testing_gate['id'] = df['id']
    cols = ['tobscene', 'tthreat', 'tinsult', 'tidentity_hate']
    if col != None and col != 'flag':
        for i, model in enumerate(models):
             testing_gate[cols[i]] = model.predict_proba(Xt)[:, 1]
    else:
        testing_gate['tflag'] = model.predict_proba(Xt)[:, 1]
    
    ndf = df.copy().merge(testing_gate, on='id')

    cols = ['obscene', 'threat', 'insult', 'identity_hate']
    if col == None:
        col = 'flag'
    if model == None:
        model = models[cols.index(col)]

    if col == 'flag':
        # Get false positive rates, true positive rates, and thresholds
        fpr, tpr, thresholds = roc_curve(ndf[col], ndf['t'+col])
        j_scores = tpr - fpr
        best_idx = j_scores.argmax()
        best_threshold = thresholds[best_idx]
        return best_threshold
    else:
        precision, recall, thresholds = precision_recall_curve(ndf[col], ndf['t' + col])
        
        # Calculate F1 score for each threshold
        # (Adding 1e-10 prevents division by zero errors)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
        
        # Find the index of the highest F1 score
        best_idx = f1_scores.argmax()
        best_threshold = thresholds[best_idx]
        if best_threshold > 0.5:
            best_threshold = 0.5
        return best_threshold
    
def test_models(gatekeeper, models, gthresh, mthresh, ndf, Xt): 
    # need the percents
    ndf['tflag'] = gatekeeper.predict_proba(Xt)[:, 1]
    cols = ['tobscene', 'tthreat', 'tinsult', 'tidentity_hate']
    for i, model in enumerate(models):
         ndf[cols[i]] = model.predict_proba(Xt)[:, 1]
    cols = ['obscene', 'threat', 'insult', 'identity_hate']
   
    false_positives = 0
    false_negatives = 0
    true_positives = 0
    true_negatives = 0
    
    for i, row in ndf.iterrows():
        if row['flag'] == 0 and row['tflag'] < gthresh:
            true_negatives +=1
        elif row['flag'] == 0 and row['tflag'] >= gthresh:
            false_positives += 1
        elif row['flag'] == 1 and row['tflag'] < gthresh:
            false_negatives += 1
        else:
            checkfn = 0
            checkfp = 0
            for col in cols:
                if row[col] == 0 and row['t'+col] >= mthresh[cols.index(col)]:
                    checkfp += 1
                elif row[col] == 1 and row['t'+col] <  mthresh[cols.index(col)]:
                    checkfn += 1
            if checkfn >= 1:
                false_negatives += 1
    
            if checkfp >= 1:
                false_positives += 1
            if checkfn == 0 and checkfp == 0:
                true_positives += 1
  
    print('TP: {} TN: {} FP: {} FN: {}'.format(true_positives, true_negatives, false_positives, false_negatives))

def train_model(Xtrain, Xt, train_df, test_df, col, verbose=True):
    
    y = train_df[col]
    yt = test_df[col]
    
    model = LogisticRegression(
        C = 0.5
        , penalty='l2'
        , solver='lbfgs'
        , class_weight='balanced'
        , n_jobs = -1
    )
    
    model.fit(Xtrain,y)

    # Calculate probabilities
    y_pred_probs = model.predict_proba(Xtrain)[:, 1]
    yt_pred_probs = model.predict_proba(Xt)[:, 1]

    thresh = 0.35
    
    # Evaluate score based on custom threshold rather than default 0.5
    y_preds = (y_pred_probs >= thresh).astype(int)
    yt_preds = (yt_pred_probs >= thresh).astype(int)
    custom_acc = (yt_preds == yt).mean()
    
    if verbose:
        print('Training Data {:.3f}, Test Data Score (Thresh={:.2f}): {:.3f}'.format(
               (y_preds == y).mean(), thresh, custom_acc
            ))
        # Calculate ROC-AUC for the single binary label
        auc_score = roc_auc_score(y, y_pred_probs)
        
        auc_scoret = roc_auc_score(yt, yt_pred_probs)
        print(f"TRAINING ROC-AUC Score: {auc_score:.4f}, Test ROC-AUC Score: {auc_scoret:.4f}")
    return model

def build_model(model_score = 0.0, 
                ogatekeeper = None, 
                omodels = None, 
                ogatekeeper_thresh = 0, 
                othresholds = [0,0,0,0,0,0], 
                onounVectorizer=None, 
                ogramVectorizer=None, 
                ocharVectorizer=None, 
                oscaler=None):

    stopword = nltk.corpus.stopwords.words('english')
    ps = nltk.PorterStemmer()

    # get and process data
    df = pd.read_csv('data/processed_data.csv')

    #toxic used here to set flag...
    cols = ['toxic', 'obscene', 'threat', 'insult', 'identity_hate']
    df['flag'] = df[cols].max(axis=1)
    df['pos'], df['stems'], df['punc'] = preprocess_text(df['comment_text'])
    
    # split the data
    train_df, test_df = train_test_split(df, test_size=0.2)
    gramVectorizer, nounVectorizer, charVectorizer, scaler = fit_vectorizers(train_df)
    
    # flag_df
    temp1 = train_df[train_df['flag'] == 1]
    # get the last 25000 comemnts in each group, focus on newer neg comments
    temp_ = pd.DataFrame()
    for col in ['toxic', 'obscene', 'threat', 'insult', 'identity_hate']:
        temp_ = pd.concat([temp_, temp1[temp1[col]==1].tail(25000)])
    print('temp_', temp_.shape)
    temp1 = temp_.drop_duplicates()
    print('temp1', temp1.shape)
   
    # twice as many non-flagged comments...
    bal_num = test_df[test_df['flag']==0].shape[0]/test_df[test_df['flag']==1].shape[0]
    temp2 = train_df[train_df['flag'] == 0].sort_values('pred', ascending=False).head(int(temp1.shape[0]*bal_num))
    flag_df = pd.concat([temp1, temp2]).sort_index().reset_index()
    tox_df = temp1.reset_index()
    test_flag_df = test_df[test_df['flag']==1].reset_index()

    # vectorize!
    Xf = vectorize_data(flag_df, nounVectorizer, gramVectorizer, charVectorizer, scaler)
    Xt = vectorize_data(test_df, nounVectorizer, gramVectorizer, charVectorizer, scaler)
    Xx = vectorize_data(tox_df, nounVectorizer, gramVectorizer, charVectorizer, scaler)
    Xtf = vectorize_data(test_flag_df, nounVectorizer, gramVectorizer, charVectorizer, scaler)

    # train the models
    gatekeeper = train_model(Xf, Xt, flag_df, test_df, col='flag', verbose=True)
    models = []
    for col in ['obscene', 'threat', 'insult', 'identity_hate']:
        models.append(train_model(Xx, Xtf, tox_df, test_flag_df, col=col, verbose=True))
        
    gatekeeper_thresh = set_threshold(model = gatekeeper, models = None, col = 'flag', df=test_df, Xt=Xt)
    thresholds = []
    [thresholds.append(set_threshold(model = None, models = models, col = col, df=test_flag_df, Xt=Xtf)) for col in ['obscene', 'threat', 'insult', 'identity_hate']]
    test_models(gatekeeper, models, gatekeeper_thresh, thresholds, ndf=test_df, Xt=Xt)

    # check score...
    scores  = []
    yt = test_df['flag']
    gscore = score_model(gatekeeper,Xt,yt,gatekeeper_thresh)

    for i, col in enumerate(['obscene', 'threat', 'insult', 'identity_hate']):
        yt = test_flag_df[col]
        scores.append(score_model(models[i],Xtf, yt,thresholds[i]))
    print(gscore, scores)
    mscore = sum(scores)/len(scores)
    # gatekeeper is more important...
    score = (gscore+mscore)/2
    print('newscore', score)
    if score >= model_score:  # = so newer models are preferred...
        #redo preds for current model
        X = vectorize_data(df, nounVectorizer, gramVectorizer, charVectorizer, scaler)
        df['pred'] = gatekeeper.predict_proba(X)[:, 1]
        cols = ['id', 'comment_text', 'toxic',	'obscene',	'threat',	'insult',	'identity_hate', 'pred']
        df[cols].to_csv('data/processed_data.csv', index=False)
        return score, gatekeeper, models, gatekeeper_thresh, thresholds, nounVectorizer, gramVectorizer, charVectorizer, scaler
    else:
        return model_score, ogatekeeper, omodels, ogatekeeper_thresh, othresholds, onounVectorizer, ogramVectorizer, ocharVectorizer, oscaler

def score_model(model, Xt, yt, thresh):
    # Calculate probabilities
    yt_pred_probs = model.predict_proba(Xt)[:, 1]
    # Evaluate score based on custom threshold rather than default 0.5
    yt_preds = (yt_pred_probs >= thresh).astype(int)
    return (yt_preds == yt).mean()

def tokenize(s):
    re_tok = re.compile(f'([{string.punctuation}])')
    # Splits words and isolates punctuation marks as individual tokens
    return re_tok.sub(r' \1 ', s).split()

def evaluate_post(post, gatekeeper, models, gatekeeper_thresh, thresholds, nounVectorizer, gramVectorizer, charVectorizer, scaler): 
    df = pd.DataFrame()
    df['comment_text'] = [post]
    df['pos'], df['stems'], df['weird'] = preprocess_text(df['comment_text'])
    X = vectorize_data(df, nounVectorizer, gramVectorizer, charVectorizer, scaler)
    cols = ['Obscene', 'Threat', 'Insult', 'Identity-Based Hate Speach']
    check = gatekeeper.predict_proba(X)[:, 1][0]
    response = []
    if check >= gatekeeper_thresh:
        response.append(('Toxic',check))
        for i, model in enumerate(models):
            check_ = model.predict_proba(X)[:, 1][0]
            if check_ > thresholds[i]:
                response.append((cols[i], check_))
            else:
                response.append((cols[i], 0))
    else:
        response = [(col, 0) for col in cols]
    return check, response

In [3]:
model_score, gatekeeper, models, gatekeeper_thresh, thresholds, nounVectorizer, gramVectorizer, charVectorizer, scaler = build_model()
print()
print(model_score)
save_model_bundle(
    'redflags_model.joblib',
    model_score, gatekeeper, models, gatekeeper_thresh, 
    thresholds, nounVectorizer, gramVectorizer, charVectorizer, scaler
)

100.00% Complete

C:\Users\darin\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


temp_ (38454, 13)
temp1 (18105, 13)
Training Data 0.924, Test Data Score (Thresh=0.35): 0.905
TRAINING ROC-AUC Score: 0.9917, Test ROC-AUC Score: 0.9692
Training Data 0.902, Test Data Score (Thresh=0.35): 0.822
TRAINING ROC-AUC Score: 0.9644, Test ROC-AUC Score: 0.9037
Training Data 0.959, Test Data Score (Thresh=0.35): 0.947
TRAINING ROC-AUC Score: 0.9966, Test ROC-AUC Score: 0.9649
Training Data 0.830, Test Data Score (Thresh=0.35): 0.702
TRAINING ROC-AUC Score: 0.9367, Test ROC-AUC Score: 0.8164
Training Data 0.902, Test Data Score (Thresh=0.35): 0.843
TRAINING ROC-AUC Score: 0.9890, Test ROC-AUC Score: 0.9069
TP: 2124 TN: 36071 FP: 5565 FN: 1013
0.8963766495191232 [0.8223699289479716, 0.9640155856062342, 0.7322942929177172, 0.9046527618611048]
newscore 0.8761048959261901

0.8761048959261901
Model bundle successfully saved to redflags_model.joblib


In [4]:
post = "That was me, loser. The UN defines Vietnam to be part of Southeast Asia. And far as I know Vietnam is part of ASEAN, and used to be part of French Indochina with Laos and all those shit countries Anyway your culture has always been more influenced by SEA than Han Chinese (as in proper Yangtze Han Chinese, not the fringe indigenous tribes in Guangzhou/Guangxi). \n\nJust admit that you vietnamese are all a bunch of wannabe crap people. ALL the east asian people I've spoken to thinks of Vietnam as a very integral part of SEA, and we all think you're backward, dirty and speak in a horrible swearing language. Doesn't matter what crap you spout on Wikipedia, won't change the way people in the real world think."



evaluate_post(post, gatekeeper, models, gatekeeper_thresh, thresholds, nounVectorizer, gramVectorizer, charVectorizer, scaler)

100.00% Complete

(0.9224722827153667,
 [('Toxic', 0.9224722827153667),
  ('Obscene', 0),
  ('Threat', 0),
  ('Insult', 0.49977956301906173),
  ('Identity-Based Hate Speach', 0)])

In [5]:
(model_score, gatekeeper, models, gatekeeper_thresh, 
 thresholds, nounVectorizer, gramVectorizer, charVectorizer, scaler) = load_model_bundle('redflags_model.joblib')

Model bundle loaded from redflags_model.joblib


In [12]:
model_score

0.879650423887888

In [17]:
post = "trannies are sins against god"
evaluate_post(post, gatekeeper, models, gatekeeper_thresh, thresholds, nounVectorizer, gramVectorizer, charVectorizer, scaler)

100.00% Complete

(0.6149591979358751,
 [('Toxic', 0.6149591979358751),
  ('Obscene', 0),
  ('Threat', 0),
  ('Insult', 0),
  ('Identity-Based Hate Speach', 0.5805580556718094)])

In [14]:
model_score, gatekeeper, models, gatekeeper_thresh, thresholds, nounVectorizer, gramVectorizer, charVectorizer, scaler = build_model(model_score, gatekeeper, models, gatekeeper_thresh, thresholds, nounVectorizer, gramVectorizer, charVectorizer, scaler)

100.00% Complete

C:\Users\darin\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


temp_ (38314, 12)
temp1 (18061, 12)
Training Data 0.924, Test Data Score (Thresh=0.35): 0.902
TRAINING ROC-AUC Score: 0.9919, Test ROC-AUC Score: 0.9678
Training Data 0.902, Test Data Score (Thresh=0.35): 0.811
TRAINING ROC-AUC Score: 0.9636, Test ROC-AUC Score: 0.8971
Training Data 0.960, Test Data Score (Thresh=0.35): 0.945
TRAINING ROC-AUC Score: 0.9962, Test ROC-AUC Score: 0.9699
Training Data 0.830, Test Data Score (Thresh=0.35): 0.705
TRAINING ROC-AUC Score: 0.9370, Test ROC-AUC Score: 0.8207
Training Data 0.895, Test Data Score (Thresh=0.35): 0.852
TRAINING ROC-AUC Score: 0.9880, Test ROC-AUC Score: 0.9203
TP: 2158 TN: 36646 FP: 4756 FN: 1222
0.9083202862894207 [0.8230088495575221, 0.9636941229861584, 0.7469934195597913, 0.9092353074653959]
newscore 0.8845266055908189


In [16]:
model_score

0.8845266055908189

In [27]:
post = "fuck yo couch"
evaluate_post(post, gatekeeper, models, gatekeeper_thresh, thresholds, nounVectorizer, gramVectorizer, charVectorizer, scaler)

100.00% Complete

(0.9999719702909831,
 [('Toxic', 0.9999719702909831),
  ('Obscene', 0.9983582505169313),
  ('Threat', 0),
  ('Insult', 0.7767374856585093),
  ('Identity-Based Hate Speach', 0)])